# 5.6 — Random Forest Regression
### Averaging Many Trees to Cancel Errors

---

## From Single Tree to Forest

Decision Tree Regressor had one big weakness — **instability**. Change a few rows in training data and you get a completely different tree. One tree overfits to specific patterns it saw during training.

Random Forest fixes this by building **hundreds of different trees** and averaging their predictions.

The key insight: **if each tree makes different errors, averaging cancels those errors out.**

---

## Two Sources of Randomness — Why Trees Are Different

### 1. Row Randomness — Bagging (Bootstrap Aggregating)
Each tree trains on a **random bootstrap sample** of the training data:
- Sample n rows **with replacement** from the training set
- ~63.2% unique rows per tree (some rows appear multiple times, some not at all)
- The ~36.8% rows NOT selected = **Out-Of-Bag (OOB)** rows for that tree

### 2. Column Randomness — Feature Subsampling
At **every split** in every tree, only a random subset of features is considered:
- For regression: default = `max_features = n_features / 3` (or `'sqrt'`)
- Tree 1 might split on `[temperature, rainfall, rating]`
- Tree 2 might split on `[hotels, festival_count, season]`
- Different splits → different tree structures → different errors

These two together ensure **maximum diversity** across trees.

---

## Final Prediction — Average (not Vote)

| | Random Forest Classifier | Random Forest Regressor |
|---|---|---|
| Combines trees by | Majority vote | **Average of all predictions** |
| Output | Class label | Mean of all trees' predicted numbers |

If 100 trees predict revenues of ₹42L, ₹38L, ₹44L... → final prediction = mean of all 100.

---

## OOB Score — Free Validation

Each tree only sees ~63.2% of training rows. The remaining ~36.8% (OOB rows) were never used to train that tree.

For each training row: find all trees that did NOT train on it → average their predictions → compare to actual value.

This gives a reliable performance estimate **without touching the test set** — free validation built into the training process.

Set `oob_score=True` in sklearn to get this automatically.

---

## Decision Tree vs Random Forest — When to Use Which

| Situation | Prefer |
|-----------|--------|
| Need to explain every prediction in plain language | Decision Tree |
| Very small dataset (< 100 rows) | Decision Tree |
| Maximum accuracy matters | **Random Forest** |
| Complex feature interactions | **Random Forest** |
| Need stable, reliable predictions | **Random Forest** |
| Predicting millions of rows fast | Decision Tree (one tree faster than 100) |

**Golden rule:** Decision Tree = interpretability. Random Forest = accuracy.

---

## Key Hyperparameters

| Parameter | What it controls | Default |
|-----------|-----------------|--------|
| `n_estimators` | Number of trees | 100 |
| `max_depth` | Max depth of each tree | None (unlimited) |
| `min_samples_leaf` | Min samples per leaf | 1 |
| `max_features` | Features considered per split | `'sqrt'` or `1/3` |
| `oob_score` | Whether to compute OOB score | False |
| `n_jobs` | CPU cores to use | 1 (-1 = all) |

More trees = better but diminishing returns after ~100-300. After that, accuracy barely improves but training time keeps growing.

---

## Real World Problem — Kerala Tourism Revenue Prediction

**Priya** works at the Kerala Tourism Development Corporation. She wants to predict **monthly revenue (₹ crores)** from tourist destinations to help plan infrastructure and staffing.

Features:
- `avg_temperature` — average temperature that month (°C)
- `rainfall_mm` — rainfall received
- `num_hotels` — number of hotels in the area
- `festival_count` — number of festivals/events that month
- `distance_from_airport_km` — accessibility
- `google_rating` — average rating (1-5)
- `prev_month_visitors` — visitors last month (momentum)
- `is_peak_season` — 0 or 1

Revenue is complex and non-linear — peak season spikes, heavy rainfall kills tourism, festivals boost it unpredictably. Random Forest handles these interactions robustly.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# WHY each import:
# RandomForestRegressor — sklearn's RF implementation
#   Internally: builds n_estimators trees in parallel using joblib
#   Each tree: bootstrap sample of rows + random feature subset at each split
#   Prediction: averages all trees' outputs
# DecisionTreeRegressor — for comparison (single tree baseline)
# GridSearchCV          — finds best n_estimators + max_depth + min_samples_leaf

np.random.seed(42)

In [ ]:
# ── Step 1: Create Kerala Tourism Dataset ─────────────────────────────────────
n = 800  # 800 destination-months

avg_temperature          = np.random.uniform(20, 38, n)
rainfall_mm              = np.random.exponential(scale=80, size=n).clip(0, 400)
# WHY exponential for rainfall? — most months have low rain, few have heavy rain
num_hotels               = np.random.randint(5, 100, n)
festival_count           = np.random.randint(0, 8, n)
distance_from_airport_km = np.random.uniform(10, 200, n)
google_rating            = np.random.uniform(3.0, 5.0, n).round(1)
prev_month_visitors      = np.random.randint(1000, 50000, n)
is_peak_season           = np.random.choice([0, 1], n, p=[0.6, 0.4])

# TRUE revenue formula — complex non-linear interactions
noise = np.random.normal(0, 1.5, n)

revenue_crores = (
    0.05  * prev_month_visitors / 1000        +  # momentum effect
    0.8   * num_hotels                        +  # more hotels = more capacity
    2.5   * festival_count                    +  # festivals boost revenue
    5.0   * google_rating                     +  # rating effect
   -0.03  * distance_from_airport_km          +  # far from airport = less visitors
   -0.08  * rainfall_mm                       +  # rain kills tourism
    8.0   * is_peak_season                    +  # peak season spike
    0.1   * (avg_temperature - 30).clip(0)**2 +  # very hot = bad (non-linear)
    10.0                                      +  # base revenue
    noise
).clip(1, 100)

df = pd.DataFrame({
    'avg_temperature':          np.round(avg_temperature, 1),
    'rainfall_mm':              np.round(rainfall_mm, 1),
    'num_hotels':               num_hotels,
    'festival_count':           festival_count,
    'distance_from_airport_km': np.round(distance_from_airport_km, 1),
    'google_rating':            google_rating,
    'prev_month_visitors':      prev_month_visitors,
    'is_peak_season':           is_peak_season,
    'revenue_crores':           np.round(revenue_crores, 2)
})

print(f"Dataset shape: {df.shape}")
print(f"Revenue range: ₹{df['revenue_crores'].min():.2f}Cr to ₹{df['revenue_crores'].max():.2f}Cr")
print(f"Average monthly revenue: ₹{df['revenue_crores'].mean():.2f} crores")
df.head()

In [ ]:
# ── Step 2: Split Data ────────────────────────────────────────────────────────
X = df.drop('revenue_crores', axis=1)
y = df['revenue_crores']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Training: {X_train.shape[0]} records | Test: {X_test.shape[0]} records")

In [ ]:
# ── Step 3: What Happens Inside RandomForestRegressor.fit() ───────────────────
#
# Step-by-step internals of .fit(X_train, y_train):
#
# 1. For each of n_estimators trees (run in parallel if n_jobs != 1):
#
#    a. BOOTSTRAP SAMPLING:
#       sample_indices = np.random.choice(n_train, size=n_train, replace=True)
#       X_boot = X_train[sample_indices]  # ~63.2% unique rows
#       y_boot = y_train[sample_indices]
#       oob_indices = rows NOT in sample_indices  # ~36.8%
#
#    b. BUILD DECISION TREE on (X_boot, y_boot):
#       At each node — instead of trying ALL features:
#         features_to_try = random.sample(all_features, k=max_features)
#         Find best split among only these features
#       This is the ONLY difference from a single DecisionTreeRegressor
#
#    c. STORE OOB predictions:
#       For each oob row: store this tree's prediction for it
#
# 2. OOB Score calculation (if oob_score=True):
#    For each training row i:
#      Find all trees where row i was OOB (not used in bootstrap)
#      Average their predictions for row i
#    OOB score = R² between these OOB predictions and actual y_train values
#
# What .predict(X_test) does:
#    For each row in X_test:
#      Run through ALL n_estimators trees
#      Collect each tree's leaf mean prediction
#      Return the average of all n_estimators predictions

rf = RandomForestRegressor(
    n_estimators=100,     # build 100 trees
    oob_score=True,       # compute free OOB validation score
    n_jobs=-1,            # use all CPU cores — trees built in parallel
    random_state=42
)
rf.fit(X_train, y_train)

y_pred    = rf.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2   = r2_score(y_test, y_pred)

print("Random Forest (100 trees, default params):")
print(f"  OOB Score (R²) : {rf.oob_score_:.4f}  ← free validation, no test set used")
print(f"  Test RMSE      : ₹{test_rmse:.2f} crores")
print(f"  Test R²        : {test_r2:.4f}")
print(f"\n.oob_score_ is computed using only training data.")
print(f"It's a reliable estimate of generalisation without touching the test set.")

In [ ]:
# ── Step 4: Effect of n_estimators — Diminishing Returns ─────────────────────
#
# More trees = more stable predictions but diminishing returns.
# After ~100-300 trees, adding more barely helps.
# Training time grows linearly with n_estimators.

n_trees_list = [1, 5, 10, 20, 50, 100, 200, 300, 500]
oob_scores   = []
test_rmses   = []

for n_trees in n_trees_list:
    rf_temp = RandomForestRegressor(
        n_estimators=n_trees,
        oob_score=True,
        n_jobs=-1,
        random_state=42
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    test_rmses.append(np.sqrt(mean_squared_error(y_test, rf_temp.predict(X_test))))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(n_trees_list, oob_scores, 'o-', color='steelblue', linewidth=2)
axes[0].set_xlabel('Number of Trees (n_estimators)')
axes[0].set_ylabel('OOB Score (R²)')
axes[0].set_title('OOB Score vs Number of Trees\n(Diminishing returns after ~100)')
axes[0].axvline(x=100, color='red', linestyle='--', linewidth=1, label='n=100')
axes[0].legend()

axes[1].plot(n_trees_list, test_rmses, 'o-', color='tomato', linewidth=2)
axes[1].set_xlabel('Number of Trees (n_estimators)')
axes[1].set_ylabel('Test RMSE (₹ crores)')
axes[1].set_title('Test RMSE vs Number of Trees\n(Stabilises after ~100)')
axes[1].axvline(x=100, color='red', linestyle='--', linewidth=1, label='n=100')
axes[1].legend()

plt.tight_layout()
plt.show()

# WHY this plot?
# Shows exactly why n_estimators=100 is a good default.
# 1 tree: high variance, unstable
# 10-50 trees: rapidly improving
# 100+ trees: plateau — more trees waste computation

In [ ]:
# ── Step 5: GridSearchCV — Tune max_depth and min_samples_leaf ────────────────
#
# GridSearchCV internally:
# 1. Creates all combinations from param_grid
#    4 depths × 4 leaf sizes × 3 n_estimators = 48 combinations
# 2. For each combination: runs cv=5 fold CV on X_train
#    48 × 5 = 240 model fits total
# 3. Picks combination with best average CV score
# 4. Retrains on full X_train with best combination
# 5. Stores as .best_estimator_
#
# WHY tune max_depth for Random Forest?
# Unlike single Decision Tree, Random Forest is less prone to overfitting
# because averaging reduces variance. But limiting depth still helps:
# - Faster training (shallower trees)
# - Sometimes better generalisation
# - Many practitioners use max_depth=None (unlimited) for RF

param_grid = {
    'n_estimators':    [50, 100, 200],
    'max_depth':       [None, 5, 10, 15],
    'min_samples_leaf': [1, 2, 5, 10]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(oob_score=False, n_jobs=-1, random_state=42),
    # WHY oob_score=False here?
    # GridSearchCV runs its own CV — OOB score would be redundant and slow things down
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1  # WHY verbose=1? — prints progress so we know it's running
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:<20}: {value}")
print(f"Best CV RMSE: ₹{-grid_search.best_score_:.4f} crores")

In [ ]:
# ── Step 6: Evaluate Best Model ───────────────────────────────────────────────
best_rf = grid_search.best_estimator_
# .best_estimator_ = RandomForestRegressor retrained on full X_train
# with best hyperparameters — ready to predict

y_pred_best = best_rf.predict(X_test)
best_rmse   = np.sqrt(mean_squared_error(y_test, y_pred_best))
best_mae    = mean_absolute_error(y_test, y_pred_best)
best_r2     = r2_score(y_test, y_pred_best)

print(f"Best Random Forest Performance:")
print(f"  RMSE : ₹{best_rmse:.2f} crores")
print(f"  MAE  : ₹{best_mae:.2f} crores  ← on average off by this much")
print(f"  R²   : {best_r2:.4f}  ← explains {best_r2*100:.1f}% of revenue variation")

In [ ]:
# ── Step 7: Feature Importance ────────────────────────────────────────────────
#
# RandomForestRegressor.feature_importances_ internally:
# 1. For EACH tree: compute feature importances using MSE reduction
#    (same as single Decision Tree — weighted MSE reduction per feature)
# 2. AVERAGE the importances across ALL trees
# 3. Normalise so they sum to 1.0
#
# WHY average across trees?
# Single tree importances are noisy — depend on which rows that tree saw.
# Averaging across 100+ trees gives a much more stable, reliable importance score.
# This is one of Random Forest's key advantages over single Decision Tree.

importances = best_rf.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature':    X.columns,
    'Importance': importances
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(9, 5))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(feat_imp_df)))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color=colors)
plt.xlabel('Feature Importance (avg MSE reduction across all trees)')
plt.title('Which Factors Drive Kerala Tourism Revenue?')
plt.tight_layout()
plt.show()

print("Feature importances (sum = 1.0):")
for feat, imp in zip(feat_imp_df['Feature'][::-1], feat_imp_df['Importance'][::-1]):
    bar = '█' * int(imp * 60)
    print(f"  {feat:<28}: {imp:.4f}  {bar}")

In [ ]:
# ── Step 8: Full Model Comparison ─────────────────────────────────────────────
# Compare all approaches on the same test set

# Linear Regression baseline
lr_pipe = Pipeline([('scaler', StandardScaler()), ('lr', LinearRegression())])
lr_pipe.fit(X_train, y_train)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pipe.predict(X_test)))
lr_r2   = r2_score(y_test, lr_pipe.predict(X_test))

# Single Decision Tree (tuned)
dt = DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=42)
dt.fit(X_train, y_train)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt.predict(X_test)))
dt_r2   = r2_score(y_test, dt.predict(X_test))

# Random Forest (default 100 trees)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))
rf_r2   = r2_score(y_test, rf.predict(X_test))

print("=" * 58)
print("MODEL COMPARISON — KERALA TOURISM REVENUE")
print("=" * 58)
print(f"{'Model':<32} {'RMSE':>10} {'R²':>10}")
print("-" * 58)
print(f"{'Linear Regression':<32} ₹{lr_rmse:>6.2f}Cr  {lr_r2:>8.4f}")
print(f"{'Decision Tree (depth=5)':<32} ₹{dt_rmse:>6.2f}Cr  {dt_r2:>8.4f}")
print(f"{'Random Forest (100 trees)':<32} ₹{rf_rmse:>6.2f}Cr  {rf_r2:>8.4f}")
print(f"{'Random Forest (tuned)':<32} ₹{best_rmse:>6.2f}Cr  {best_r2:>8.4f}")
print("=" * 58)
improvement = ((dt_rmse - best_rmse) / dt_rmse) * 100
print(f"\nRandom Forest improved over single Decision Tree by {improvement:.1f}%")
print("Diversity of trees cancels individual errors — classic ensemble effect.")

In [ ]:
# ── Step 9: Visualise Predictions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='steelblue', s=20)
axes[0].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Revenue (₹ crores)')
axes[0].set_ylabel('Predicted Revenue (₹ crores)')
axes[0].set_title('Random Forest — Actual vs Predicted')
axes[0].legend()

residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color='tomato', s=20)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Predicted Revenue (₹ crores)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Residual Plot\n(Smoother than single tree — averaging effect)')

plt.tight_layout()
plt.show()

# WHY smoother residuals than Decision Tree?
# Single tree: discrete predictions (one mean per leaf) → staircase residuals
# Random Forest: average of 100 trees' means → continuous-looking predictions
# More trees = smoother predictions = smoother residual scatter

In [ ]:
# ── Step 10: OOB Score vs Test Score — How Reliable is OOB? ──────────────────
#
# OOB score is computed purely on training data — no test set involvement.
# How close is it to the actual test score?

rf_oob = RandomForestRegressor(
    n_estimators=200,
    oob_score=True,
    n_jobs=-1,
    random_state=42,
    **{k: v for k, v in grid_search.best_params_.items() if k != 'n_estimators'}
)
rf_oob.fit(X_train, y_train)

oob_r2  = rf_oob.oob_score_
test_r2_oob = r2_score(y_test, rf_oob.predict(X_test))

print("OOB Score vs Test Score (both R²):")
print(f"  OOB R²  : {oob_r2:.4f}  ← computed on training data only (free)")
print(f"  Test R² : {test_r2_oob:.4f}  ← computed on held-out test data")
print(f"  Gap     : {abs(oob_r2 - test_r2_oob):.4f}")
print("\nSmall gap = OOB score is a reliable estimate of test performance.")
print("This is why OOB score is valuable — reliable validation without a separate val set.")

In [ ]:
# ── Step 11: Priya's Revenue Forecast for Specific Scenarios ─────────────────
scenarios = pd.DataFrame({
    'avg_temperature':          [25,  32,  27,  35,  24],
    'rainfall_mm':              [20,  5,   200, 10,  50],
    'num_hotels':               [50,  80,  30,  20,  60],
    'festival_count':           [3,   1,   0,   5,   2],
    'distance_from_airport_km': [30,  15,  80,  120, 25],
    'google_rating':            [4.5, 4.2, 3.8, 4.7, 4.0],
    'prev_month_visitors':      [30000, 45000, 10000, 5000, 25000],
    'is_peak_season':           [1,   1,   0,   0,   1]
})

scenario_labels = [
    'Peak season, good weather',
    'Peak season, near airport',
    'Off season, heavy rain',
    'Remote, festival boost',
    'Peak season, average'
]

revenue_forecasts = best_rf.predict(scenarios)

print("Priya's Monthly Revenue Forecasts:")
print("-" * 55)
for label, rev in zip(scenario_labels, revenue_forecasts):
    bar = '█' * int(rev / 3)
    print(f"  {label:<30}: ₹{rev:>5.1f}Cr  {bar}")

---

## Internal Functions — Quick Reference

| Function / Attribute | What it does internally |
|---|---|
| `.fit(X, y)` | Builds n_estimators trees in parallel. Each tree: bootstrap rows + random feature subset at each split |
| `.predict(X)` | Runs each row through all n_estimators trees, returns average of all leaf means |
| `.oob_score_` | R² computed on OOB rows — each training row evaluated by trees that didn't train on it |
| `.feature_importances_` | Average of per-tree MSE reduction importances across all trees, normalised to 1.0 |
| `.estimators_` | List of all individual DecisionTreeRegressor objects — you can inspect any tree |
| `n_jobs=-1` | Uses all CPU cores — trees built in parallel via joblib |
| `GridSearchCV.best_estimator_` | RF retrained on full X_train with best found params |
| `verbose=1` | Prints progress during GridSearchCV fitting |

---

## Summary Table

| | Random Forest Regressor |
|---|---|
| **Task** | Regression — complex, non-linear, many features |
| **How it works** | Builds n trees on bootstrap samples with random feature subsets, averages predictions |
| **Randomness sources** | Row (bootstrap) + Column (random feature subset per split) |
| **Prediction** | Mean of all trees' leaf predictions |
| **OOB score** | Free validation using the ~36.8% rows each tree didn't see |
| **Key hyperparameters** | `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` |
| **n_estimators sweet spot** | 100-300 — diminishing returns after that |
| **Needs scaling?** | ❌ No — tree splits use thresholds, not distances |
| **Feature importance** | Average MSE reduction per feature across all trees — more stable than single tree |
| **Strength** | High accuracy, stable, handles non-linearity, built-in OOB validation |
| **Weakness** | Not interpretable (can't print 100 trees), slower than single tree |
| **When to use** | Complex regression where accuracy matters more than interpretability |

---

## What's Next?

Random Forest builds trees **in parallel** — each tree independent of the others.

**5.7 Gradient Boosting / XGBoost** takes a different approach — it builds trees **sequentially**, where each new tree specifically targets and fixes the errors made by the previous trees. This makes it even more powerful than Random Forest on structured/tabular data.

---

## Practice Task

Suresh manages a chain of petrol bunks across Tamil Nadu. He wants to predict **daily fuel sales (litres)** based on:
- `location_traffic` — vehicles passing per hour
- `num_competitors_nearby` — competing stations within 2km
- `price_per_litre` — current fuel price
- `day_of_week` — 1 to 7
- `is_highway` — 0 or 1 (highway vs city)
- `temperature_c` — outside temperature
- `bunk_age_years` — how old the station is

**Your tasks:**

1. Create synthetic dataset of 1000 days with non-linear fuel sales patterns
2. Train Random Forest with `oob_score=True` — compare OOB score vs test R²
3. Plot the n_estimators vs test RMSE curve — find the sweet spot
4. Use GridSearchCV to tune `max_depth` and `min_samples_leaf`
5. Plot feature importances — which factor drives fuel sales most?
6. Compare: Linear Regression vs Decision Tree vs Random Forest

In [ ]:
# YOUR CODE HERE

# Step 1: Create dataset

# Step 2: Random Forest with oob_score=True

# Step 3: n_estimators vs RMSE curve

# Step 4: GridSearchCV

# Step 5: Feature importances

# Step 6: Full model comparison